In [23]:
import csv
import json
import shutil
import re

from pathlib import Path
from collections import Counter

ENGINEER = "Olermo"
STUDENT_ID = "TUPM-26-1925"
SEED_DIGIT = 5

TARGET_YEAR = 2015 + SEED_DIGIT

PROJECT_NAME = "Lab6_File_Handling_Additional_Exercises"

cwd = Path.cwd()

if cwd.name == PROJECT_NAME:
    PROJECT_DIR = cwd
elif cwd.name.startswith("Exercise") and cwd.parent.name == PROJECT_NAME:
    PROJECT_DIR = cwd.parent
elif (cwd / PROJECT_NAME / "data").is_dir():
    PROJECT_DIR = cwd / PROJECT_NAME
else:
    PROJECT_DIR = next(
        (p for p in cwd.parents if p.name == PROJECT_NAME),
        None
    )

if PROJECT_DIR is None:
    raise FileNotFoundError("Project folder not found.")

EXERCISE_DIR = PROJECT_DIR / "Exercise3_Cybersecurity"

DATA_FILE = (
    PROJECT_DIR
    / "data"
    / "Global_Cybersecurity_Threats_2015-2024.csv"
)

OUTPUT_DIR = EXERCISE_DIR / "output"
BACKUP_DIR = EXERCISE_DIR / "backup"
REPORT_DIR = EXERCISE_DIR / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_FILE)
print("Dataset exists:", DATA_FILE.exists())

print("Target year:", TARGET_YEAR)

Dataset: d:\Downloads\OLERMO\Lab6_File_Handling_Olermo\Lab6_File_Handling_Additional_Exercises\data\Global_Cybersecurity_Threats_2015-2024.csv
Dataset exists: True
Target year: 2020


In [13]:
def normalize_column(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def find_column(headers, possible_names):

    normalized = {
        normalize_column(header): header
        for header in headers
    }

    for name in possible_names:

        key = normalize_column(name)

        if key in normalized:
            return normalized[key]

    raise ValueError(
        f"Column not found: {possible_names}"
    )


def convert_number(value):

    try:
        cleaned = str(value).replace(",", "").replace("$", "").strip()
        return float(cleaned)

    except (ValueError, TypeError):
        return None

In [14]:
selected_incidents = []

attack_counts = Counter()

total_records = 0
invalid_losses = 0

financial_losses = []

try:

    with open(DATA_FILE, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        headers = reader.fieldnames

        year_col = find_column(headers, ["Year"])
        attack_col = find_column(headers, ["Attack Type"])
        loss_col = find_column(
            headers,
            ["Financial Loss (in Million $)", "Financial Loss"]
        )

        for row in reader:

            total_records += 1

            year = convert_number(row.get(year_col))

            if year != TARGET_YEAR:
                continue

            selected_incidents.append(row)

            attack_type = str(
                row.get(attack_col, "")
            ).strip()

            if not attack_type:
                attack_type = "Unknown"

            attack_counts[attack_type] += 1

            loss = convert_number(row.get(loss_col))

            if loss is not None:
                financial_losses.append(loss)

            else:
                invalid_losses += 1

except FileNotFoundError:
    print("Dataset not found:", DATA_FILE)

except Exception as error:
    print("Error:", error)

In [15]:
total_loss = sum(financial_losses)

average_loss = (
    total_loss / len(financial_losses)
    if financial_losses else None
)

print("Total records:", total_records)
print("Incidents in target year:", len(selected_incidents))
print("Count per attack type:", dict(attack_counts))
print("Total financial loss:", total_loss, "million USD")
print("Average financial loss:", average_loss, "million USD")
print("Invalid financial values:", invalid_losses)

Total records: 3000
Incidents in target year: 315
Count per attack type: {'Malware': 37, 'Phishing': 61, 'Man-in-the-Middle': 47, 'DDoS': 56, 'SQL Injection': 55, 'Ransomware': 59}
Total financial loss: 15767.95 million USD
Average financial loss: 50.056984126984126 million USD
Invalid financial values: 0


In [16]:
ranked_incidents = []

for original_index, row in enumerate(selected_incidents):

    loss = convert_number(row.get(loss_col))

    if loss is not None:

        ranked_incidents.append({
            "original_index": original_index,
            "financial_loss": loss,
            "record": row
        })


ranked_incidents.sort(
    key=lambda item: item["financial_loss"],
    reverse=True
)

top_five = ranked_incidents[:5]

print("FIVE COSTLIEST INCIDENTS")

for rank, incident in enumerate(top_five, start=1):

    print(
        rank,
        incident["financial_loss"],
        "million USD"
    )

FIVE COSTLIEST INCIDENTS
1 99.9 million USD
2 99.83 million USD
3 99.81 million USD
4 99.72 million USD
5 99.64 million USD


In [17]:
filtered_csv = OUTPUT_DIR / "filtered_cybersecurity_incidents.csv"

if selected_incidents:

    with open(filtered_csv, "w", encoding="utf-8", newline="") as file:

        writer = csv.DictWriter(
            file,
            fieldnames=selected_incidents[0].keys()
        )

        writer.writeheader()
        writer.writerows(selected_incidents)

    print("Filtered CSV created.")

else:
    print("No incidents found for the target year.")

Filtered CSV created.


In [18]:
top_five_data = []

for rank, incident in enumerate(top_five, start=1):

    top_five_data.append({
        "rank": rank,
        "financial_loss_million_usd": incident["financial_loss"],
        "incident": incident["record"]
    })


top_five_json = OUTPUT_DIR / "top_five_costliest_incidents.json"

with open(top_five_json, "w", encoding="utf-8") as file:

    json.dump(top_five_data, file, indent=4)

print("Top five incidents exported to JSON.")

Top five incidents exported to JSON.


In [19]:
summary = {
    "student": ENGINEER,
    "student_id": STUDENT_ID,
    "target_year": TARGET_YEAR,
    "total_dataset_records": total_records,
    "incidents_in_target_year": len(selected_incidents),
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_million_usd": total_loss,
    "average_financial_loss_million_usd": average_loss,
    "valid_financial_values": len(financial_losses),
    "invalid_financial_values": invalid_losses,
    "five_costliest_incidents": top_five_data
}

json_file = OUTPUT_DIR / "cybersecurity_summary.json"

with open(json_file, "w", encoding="utf-8") as file:

    json.dump(summary, file, indent=4)

print("JSON summary created.")

JSON summary created.


In [20]:
backup_file = BACKUP_DIR / "cybersecurity_summary_backup.json"

shutil.copy2(json_file, backup_file)

if backup_file.exists():

    with open(json_file, "rb") as original:
        original_content = original.read()

    with open(backup_file, "rb") as backup:
        backup_content = backup.read()

    if original_content == backup_content:
        print("JSON backup verified successfully.")
    else:
        print("JSON backup verification failed.")

else:
    print("Backup file does not exist.")

JSON backup verified successfully.


In [21]:
report_file = REPORT_DIR / "cybersecurity_incident_report.txt"

with open(report_file, "w", encoding="utf-8") as file:

    file.write("CYBERSECURITY INCIDENT REPORT\n")
    file.write("=" * 40 + "\n\n")

    file.write(f"Student: {ENGINEER}\n")
    file.write(f"Student ID: {STUDENT_ID}\n")
    file.write(f"Target Year: {TARGET_YEAR}\n\n")

    file.write(f"Total Dataset Records: {total_records}\n")
    file.write(f"Incidents in Target Year: {len(selected_incidents)}\n\n")

    file.write("COUNT PER ATTACK TYPE\n")

    for attack, count in attack_counts.items():
        file.write(f"{attack}: {count}\n")

    file.write("\nFINANCIAL LOSS\n")
    file.write(f"Total: {total_loss} million USD\n")
    file.write(f"Average: {average_loss} million USD\n")
    file.write(f"Invalid values: {invalid_losses}\n")

    file.write("\nFIVE COSTLIEST INCIDENTS\n")

    for incident in top_five:
        file.write(
            f"{incident['financial_loss']} million USD\n"
        )

    file.write("\nOUTPUT FILES\n")
    file.write(f"Filtered CSV: {filtered_csv}\n")
    file.write(f"JSON Summary: {json_file}\n")
    file.write(f"JSON Backup: {backup_file}\n")

print("Text report created successfully.")

Text report created successfully.


In [22]:
print("\n===== CYBERSECURITY INCIDENT REPORT =====")

print("Target year:", TARGET_YEAR)
print("Total dataset records:", total_records)
print("Incidents in target year:", len(selected_incidents))
print("Count per attack type:", dict(attack_counts))
print("Total financial loss:", total_loss)
print("Average financial loss:", average_loss)
print("Five costliest incidents:", len(top_five))
print("Filtered CSV:", filtered_csv)
print("JSON summary:", json_file)
print("Backup verified:", backup_file.exists())
print("Report file:", report_file)


===== CYBERSECURITY INCIDENT REPORT =====
Target year: 2020
Total dataset records: 3000
Incidents in target year: 315
Count per attack type: {'Malware': 37, 'Phishing': 61, 'Man-in-the-Middle': 47, 'DDoS': 56, 'SQL Injection': 55, 'Ransomware': 59}
Total financial loss: 15767.95
Average financial loss: 50.056984126984126
Five costliest incidents: 5
Filtered CSV: output\filtered_cybersecurity_incidents.csv
JSON summary: output\cybersecurity_summary.json
Backup verified: True
Report file: reports\cybersecurity_incident_report.txt
